# MouseFlow — Face Camera Analysis Notebook

**Author:** Oliver Barnstedt | **Adapted for local Windows use by:** Cemal Akmese (Diester/Bödecker Lab, University of Freiburg)

This notebook demonstrates the core MouseFlow pipeline for quantifying mouse facial movement from a close-up face camera video:

1. **Setup** — imports, paths, and DLC inference
2. **Inspect data** — load and preview DLC marker output
3. **Pupil diameter** — extract pupil size from 6 pupil markers
4. **Motion energy** — pixel-wise frame differences for global movement
5. **Face regions** — anatomical masks from facial landmarks
6. **Optical flow** — directional movement via Farneback algorithm
7. **Frequency analysis** — whisking/sniffing frequency via sine fitting

---
### ⚠️ Before you start

**Requirements (install once in your `mouseflow-dlc` conda env):**
```bash
conda activate mouseflow-dlc
pip install git+https://github.com/obarnstedt/MouseFlow
conda install -c conda-forge ffmpeg
pip install flow_vis bokeh==2.4.3 pandas-bokeh imageio
```

**You need:**
- A face camera video (`.mp4`) — close-up side view of mouse face
- The MouseFace DLC model folder at your `models_dir`
- Run **Section 1** first to generate the `.h5` marker file, then proceed in order


---
## 1. Setup — Imports, Paths, and DLC Inference

### 1.1 All imports and helper functions

Run this cell **once at the start of every session**. It imports all required libraries and defines helper functions used throughout the notebook.


In [1]:
# ── Core imports ──────────────────────────────────────────────────────────
import os
import glob
import math
import cv2
import numpy as np
import pandas as pd
import imageio
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import cm
from base64 import b64encode
from IPython.display import HTML
from tqdm import tqdm
from scipy.stats import zscore
from scipy import signal, optimize
import pandas_bokeh
from bokeh.plotting import curdoc
import flow_vis

pandas_bokeh.output_notebook()

# ── Smoothing function (Hanning-window convolution) ───────────────────────
def smooth(x, window_len=11, window='hanning'):
    """Smooth a 1D signal using a window function.
    
    Args:
        x          : input signal (array-like)
        window_len : number of frames in smoothing window (odd integer)
        window     : window type — 'flat' (moving average), 'hanning', 'hamming', 'bartlett', 'blackman'
    Returns:
        smoothed signal (same length as x)
    """
    if window_len < 3:
        return x
    s = np.r_[x[window_len-1:0:-1], x, x[-2:-window_len-1:-1]]
    w = np.ones(window_len, 'd') if window == 'flat' else eval('np.' + window + '(window_len)')
    y = np.convolve(w / w.sum(), s, mode='valid')
    return y

# ── Inline video display helper ────────────────────────────────────────────
def display_video(frames, figsize=(6, 4), fps=30):
    """Display a list of frames as an inline HTML5 animation."""
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')
    im = ax.imshow(frames[0])
    def update(i):
        im.set_data(frames[i])
        return [im]
    return animation.FuncAnimation(fig, update, frames=len(frames), interval=1000 / fps)

print("✅ All imports and helpers loaded successfully.")


Loading BokehJS ...

✅ All imports and helpers loaded successfully.


### 1.2 Set your file paths

**Edit the paths below to match your setup before running.**

- `models_dir` — folder containing the MouseFace DLC model
- `facevidpath` — your face camera video file
- `out_dir` — output folder for DLC marker files and videos
- `save_videos` — set to `True` to generate labeled output videos (slower), `False` to skip


In [2]:
# ── ✏️  EDIT THESE PATHS ─────────────────────────────────────────────────

# Path to the folder containing the MouseFace DLC model
models_dir = r"C:\Users\ca1055\MouseFlow_models"

# Path to your face camera video
facevidpath = r"C:\Users\ca1055\Downloads\vid3_short.mp4"

# Output directory (DLC marker .h5 files and optional labeled videos)
out_dir = r"C:\Users\ca1055\Downloads\mouseflow"

# Set to True to save labeled videos (pupil, motion energy, optical flow)
save_videos = True

# ── Auto-create output directory if it doesn't exist ──────────────────────
os.makedirs(out_dir, exist_ok=True)

# ── Verify video file ──────────────────────────────────────────────────────
cap = cv2.VideoCapture(facevidpath)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_vid   = cap.get(cv2.CAP_PROP_FPS)
width     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"✅ Video found: {os.path.basename(facevidpath)}")
print(f"   Frames: {n_frames}  |  FPS: {fps_vid}  |  Resolution: {width}x{height}")
print(f"   Duration: {n_frames/fps_vid:.1f} s")


✅ Video found: vid3_short.mp4
   Frames: 3000  |  FPS: 30.0  |  Resolution: 640x480
   Duration: 100.0 s


### 1.3 Run DLC inference

This cell runs the pre-trained MouseFace DeepLabCut model on your video and saves a marker `.h5` file to `out_dir/mouseflow/`.

**⏱ Runtime:** ~7 frames/sec on CPU. For a 3000-frame clip ≈ 7 min; for a full 30-min session ≈ 2 hours.

**Tips:**
- To shorten a long video first, run the optional trimming cell below.
- Set `overwrite=False` to skip already-processed videos.
- The `facekey` must be a string present in your video filename (e.g. `'face'`, `'vid3'`).


In [3]:
# ── Optional: trim a long video to N frames before running DLC ───────────
# Uncomment and edit if your video is very long (> 30 min)

# n_trim = 3000  # number of frames to keep
# trimmed_path = facevidpath.replace('.mp4', '_short.mp4')
# cap = cv2.VideoCapture(facevidpath)
# fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# out_trim = cv2.VideoWriter(trimmed_path, fourcc,
#                            cap.get(cv2.CAP_PROP_FPS),
#                            (int(cap.get(3)), int(cap.get(4))))
# for _ in tqdm(range(n_trim)):
#     ret, frame = cap.read()
#     if not ret: break
#     out_trim.write(frame)
# cap.release(); out_trim.release()
# print(f"Saved trimmed video to {trimmed_path}")
# facevidpath = trimmed_path  # use trimmed video for DLC

# ── Run DLC ───────────────────────────────────────────────────────────────
from mouseflow import runDLC

runDLC(
    models_dir=models_dir,
    vid_dir=os.path.dirname(facevidpath),   # folder containing your video
    facekey=os.path.basename(facevidpath).split('.')[0],  # auto-detect key from filename
    bodykey='',          # no body video
    dgp=False,           # use DLC (not DeepGraphPose)
    vid_output=False,    # skip labeled video generation here (faster)
    overwrite=False,     # skip if already processed
)


Video vid3_short.mp4 already labelled. Skipping ahead...


### 1.4 Load DLC marker file

After DLC finishes, load the resulting `.h5` file. The file contains per-frame x/y coordinates and detection likelihood for each facial marker.

**Markers tracked by MouseFace model:**
`nosetip`, `forehead`, `mouthtip`, `chin`, `tearduct`, `eyelid1`, `eyelid2`, `pupil1`–`pupil6`


In [4]:
# ── Auto-find the most recent .h5 file in out_dir ────────────────────────
h5files = sorted(glob.glob(os.path.join(out_dir, '*.h5')))
if not h5files:
    print("❌ No .h5 file found. Run DLC inference first (Section 1.3).")
else:
    dlc_facepath = h5files[-1]  # use most recent
    dlc_face = pd.read_hdf(dlc_facepath, mode='r')
    dlc_face.columns = dlc_face.columns.droplevel(0)  # drop scorer level
    sample_size = len(dlc_face)
    
    print(f"✅ Loaded: {os.path.basename(dlc_facepath)}")
    print(f"   Frames: {sample_size}")
    print(f"   Markers: {dlc_face.columns.get_level_values(0).unique().tolist()}")
    print(f"\nColumn structure (first 6):")
    print(dlc_face.iloc[0].head(6))


✅ Loaded: vid3_shortDLC_resnet50_MouseFaceAug21shuffle1_1030000.h5
   Frames: 3000
   Markers: ['nosetip', 'forehead', 'mouthtip', 'chin', 'tearduct', 'eyelid1', 'eyelid2', 'pupil1', 'pupil2', 'pupil3', 'pupil4', 'pupil5', 'pupil6']

Column structure (first 6):
bodyparts  coords    
nosetip    x             1.690705
           y             3.385817
           likelihood    0.000038
forehead   x             0.731929
           y             3.646824
           likelihood    0.000219
Name: 0, dtype: float64


---
## 2. Inspect Data

### 2.1 Show raw face video

Displays the original face camera video inline. May take ~30 seconds to load for long videos.


In [5]:
# Open raw face video in default media player
import os
os.startfile(facevidpath)

### 2.2 Show video with DLC marker points

Generates a video with all DLC markers overlaid as colored circles, then displays it inline.
Set `save_videos = True` in Section 1.2 to enable this.


In [6]:
if save_videos:
    frames_out = min(500, sample_size)
    cap = cv2.VideoCapture(facevidpath)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_vid = cv2.VideoWriter(
        os.path.join(out_dir, 'DLC_points.mp4'), fourcc, 30,
        (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    )
    markers = dlc_face.columns.get_level_values(0).unique().tolist()
    i = 0
    with tqdm(total=frames_out) as pbar:
        while cap.isOpened() and i < frames_out:
            facetime = dlc_face.loc[i]
            ret, frame = cap.read()
            if not ret:
                break
            for j, marker in enumerate(markers):
                x, y = int(facetime[marker]['x']), int(facetime[marker]['y'])
                cv2.circle(frame, (x, y), radius=8,
                           color=[k*255 for k in cm.jet(round(255/len(markers)*j))[:3]],
                           thickness=-1)
            out_vid.write(frame)
            pbar.update(1)
            i += 1
    cap.release()
    out_vid.release()
    print(f"✅ Saved DLC points video to {out_dir}/DLC_points.mp4")
    os.startfile(os.path.join(out_dir, 'DLC_points.mp4'))

100%|██████████| 500/500 [00:02<00:00, 233.57it/s]


✅ Saved DLC points video to C:\Users\ca1055\Downloads\mouseflow/DLC_points.mp4


---
## 3. Pupil Diameter

The MouseFace model tracks 6 points around the pupil (`pupil1`–`pupil6`). We fit the smallest enclosing circle to these 6 points to extract pupil centre and diameter each frame.

**Steps:**
1. Filter out low-confidence detections (likelihood < threshold)
2. Interpolate short gaps linearly
3. Fit enclosing circle → diameter in pixels
4. Smooth with Hanning filter


### 3.1 Clean DLC data by likelihood threshold

In [7]:
# ── Threshold likelihood and interpolate gaps ─────────────────────────────
like_thresh = 0.9  # markers with likelihood below this are set to NaN

dlc_face_clean = dlc_face.copy()
dlc_face_clean.loc(axis=1)[:, ['x', 'y']] = (
    np.tile([dlc_face_clean.loc(axis=1)[:, 'likelihood'].to_numpy()], 2)[0] > like_thresh
) * dlc_face_clean.loc(axis=1)[:, ['x', 'y']]
dlc_face_clean[dlc_face_clean == 0] = np.nan

# Interpolate linearly across NaN gaps, then back/forward fill ends
dlc_face_clean = dlc_face_clean.interpolate()
dlc_face_clean = dlc_face_clean.bfill().ffill()

nan_frac = dlc_face_clean.loc(axis=1)[:, 'x'].isnull().mean().mean()
print(f"✅ Cleaned. NaN fraction after interpolation: {nan_frac:.3f}")
dlc_face_clean.head(3)


✅ Cleaned. NaN fraction after interpolation: 0.308


bodyparts nosetip                  forehead                      mouthtip      \
coords          x   y likelihood          x         y likelihood        x   y   
0             NaN NaN   0.000038  83.757362  4.065792   0.000219      NaN NaN   
1             NaN NaN   0.000157  83.757362  4.065792   0.001432      NaN NaN   
2             NaN NaN   0.000024  83.757362  4.065792   0.000064      NaN NaN   

bodyparts                  chin  ...     pupil3      pupil4              \
coords    likelihood          x  ... likelihood           x           y   
0           0.000052  88.080833  ...   0.000006  486.678223  269.188721   
1           0.000080  88.080833  ...   0.000090  486.678223  269.188721   
2           0.000017  88.080833  ...   0.000005  486.678223  269.188721   

bodyparts                 pupil5                             pupil6  \
coords    likelihood           x           y likelihood           x   
0           0.000083  473.440674  280.788788   0.000060  453.409668   
1           0.000835  473.440674  280.788788   0.000197  453.409668   
2           0.000009  473.440674  280.788788   0.000025  453.409668   

bodyparts                        
coords             y likelihood  
0          279.03833   0.000006  
1          279.03833   0.000038  
2          279.03833   0.000004  

[3 rows x 39 columns]

### 3.2 Extract pupil diameter per frame

In [8]:
# ── Fit minimum enclosing circle to 6 pupil markers ──────────────────────
pupil_diam = pd.Series(index=range(sample_size), dtype=float)

for i in tqdm(range(sample_size), desc='Extracting pupil diameter'):
    facetime = dlc_face_clean.iloc[i]
    pupilpoints = np.float32(
        facetime.loc[['pupil1', 'pupil2', 'pupil3', 'pupil4', 'pupil5', 'pupil6'], ['x', 'y']]
        .values.reshape(6, 2)
    )
    if np.isnan(pupilpoints).any():
        pupil_diam.iloc[i] = np.nan
    else:
        _, radius = cv2.minEnclosingCircle(pupilpoints)
        pupil_diam.iloc[i] = radius * 2  # diameter in pixels

print(f"\n✅ Done. Mean diameter: {pupil_diam.mean():.1f} px  |  NaN: {pupil_diam.isna().mean():.2%}")


Extracting pupil diameter: 100%|██████████| 3000/3000 [00:01<00:00, 2868.81it/s]


✅ Done. Mean diameter: 40.1 px  |  NaN: 0.00%


### 3.3 Smooth and plot pupil diameter

In [9]:
# ── Smooth with Hanning filter ────────────────────────────────────────────
smooth_window = 75  # frames (= 1 sec at 75 fps); adjust to your FPS

pupil_diam_smooth = (
    pd.Series(smooth(pupil_diam.fillna(pupil_diam.median()), window_len=smooth_window))
    .shift(periods=-int(smooth_window / 2))[:sample_size]
    .astype(float)
)

# ── Plot ──────────────────────────────────────────────────────────────────
pupil_diam_df = pd.DataFrame({'raw': pupil_diam.astype(float),
                               'smoothed': pupil_diam_smooth})
pupil_diam_df.plot(figsize=(14, 4), alpha=0.7,
                   xlabel='Frame', ylabel='Pupil diameter [px]',
                   title='Pupil Diameter over Session')
plt.tight_layout()
plt.show()


### 3.4 Optional: save pupil tracking video

Generates a cropped eye-region video with the fitted pupil circle overlaid.
Only runs if `save_videos = True`.


In [10]:
if save_videos:
    frames_out_pupil = min(500, sample_size)
    cap = cv2.VideoCapture(facevidpath)
    width_v  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height_v = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    # Crop region around eye — adjust x_cut / y_cut for your video
    x_cut = (int(width_v * 0.58), int(width_v * 0.19))
    y_cut = (int(height_v * 0.60), int(height_v * 0.12))
    out_w = width_v  - x_cut[0] - x_cut[1]
    out_h = height_v - y_cut[0] - y_cut[1]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_vid = cv2.VideoWriter(os.path.join(out_dir, 'pupil.mp4'), fourcc, 30, (out_w, out_h))
    i = 0
    with tqdm(total=frames_out_pupil, desc='Saving pupil video') as pbar:
        while cap.isOpened() and i < frames_out_pupil:
            facetime = dlc_face_clean.loc[i]
            ret, frame = cap.read()
            if not ret:
                break
            pupilpoints = np.float32(
                facetime.loc[['pupil1','pupil2','pupil3','pupil4','pupil5','pupil6'], ['x','y']]
                .values.reshape(6, 2))
            if not np.isnan(pupilpoints).any():
                centre, radius = cv2.minEnclosingCircle(pupilpoints)
                cv2.circle(frame, [int(k) for k in centre], int(radius),
                           (255, 255, 255), 1, cv2.LINE_AA)
            image = frame[y_cut[1]:height_v - y_cut[0], x_cut[0]:width_v - x_cut[1]]
            image = cv2.convertScaleAbs(image, alpha=2, beta=0)
            out_vid.write(image)
            pbar.update(1)
            i += 1
    cap.release()
    out_vid.release()
    print(f"✅ Pupil video saved to {out_dir}/pupil.mp4")
    os.startfile(os.path.join(out_dir, 'pupil.mp4'))


Saving pupil video: 100%|██████████| 500/500 [00:00<00:00, 1196.78it/s]


✅ Pupil video saved to C:\Users\ca1055\Downloads\mouseflow/pupil.mp4


---
## 4. Motion Energy (Global)

Motion energy is the frame-by-frame absolute pixel difference — a simple but effective measure of how much movement occurred anywhere in the image. High motion energy = more movement.

This section calculates global motion energy (whole frame), before restricting to face regions in Section 5.


In [11]:
# ── Generate motion energy video (optional) ───────────────────────────────
frames_out_me = min(300, sample_size)
cap = cv2.VideoCapture(facevidpath)
frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if save_videos:
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_vid = cv2.VideoWriter(os.path.join(out_dir, 'motion_energy.mp4'),
                               fourcc, 30, (frame_width * 2, frame_height))
    ret, current_frame = cap.read()
    previous_frame = current_frame.copy()
    i = 0
    with tqdm(total=frames_out_me, desc='Saving motion energy video') as pbar:
        while cap.isOpened() and i < frames_out_me:
            if not ret:
                break
            gray_curr = cv2.cvtColor(current_frame,  cv2.COLOR_BGR2GRAY)
            gray_prev = cv2.cvtColor(previous_frame, cv2.COLOR_BGR2GRAY)
            diff = cv2.applyColorMap(cv2.absdiff(gray_curr, gray_prev), cv2.COLORMAP_VIRIDIS)
            cv2.putText(diff, str(i), (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
            out_vid.write(np.concatenate((diff, current_frame), axis=1))
            previous_frame = current_frame.copy()
            ret, current_frame = cap.read()
            i += 1
            pbar.update(1)
    cap.release()
    out_vid.release()
    print(f"✅ Motion energy video saved to {out_dir}/motion_energy.mp4")
else:
    cap.release()


Saving motion energy video: 100%|██████████| 300/300 [00:01<00:00, 189.19it/s]

✅ Motion energy video saved to C:\Users\ca1055\Downloads\mouseflow/motion_energy.mp4


---
## 5. Face Regions and Region-Specific Motion Energy. If you have only pupil recording, skip section 5.

We use median landmark positions to define anatomical face regions (nose, whisker pad, mouth, cheek) as elliptical/circular masks. Motion energy is then computed separately within each region.

**Note:** This section requires a full-face video with visible nose, whiskers, tearduct and chin. If your video only shows the eye region, skip to Section 6.


### 5.1 Define face region masks from landmark medians

In [14]:
# ── Compute median positions of facial anchors ────────────────────────────
face_anchor = dlc_face_clean.loc(axis=1)[
    ['nosetip', 'forehead', 'mouthtip', 'chin', 'tearduct', 'eyelid2'], ['x', 'y']
].median()

# ── Infer region centres from landmark geometry ───────────────────────────
whiskercentre = np.array(cv2.minEnclosingCircle(
    np.array(np.vstack([face_anchor.nosetip, face_anchor.mouthtip, face_anchor.tearduct]),
             dtype=np.float32))[0])
whiskercentre = tuple(np.round(whiskercentre).astype(int) + [-10, 70])
nosecentre    = tuple(np.round(face_anchor.nosetip).astype(int) + [50, 0])
mouthcentre   = tuple(np.round(face_anchor.mouthtip + (face_anchor.chin - face_anchor.mouthtip) / 3).astype(int))
mouthangle    = math.degrees(math.atan2(face_anchor.chin[1] - face_anchor.mouthtip[1],
                                         face_anchor.chin[0] - face_anchor.mouthtip[0]))
cheek_centre  = tuple(np.round(face_anchor.eyelid2 + (face_anchor.chin - face_anchor.eyelid2) / 2).astype(int))

# ── Visualise on first frame ──────────────────────────────────────────────
cap = cv2.VideoCapture(facevidpath)
firstframe = np.array(cap.read()[1][:, :, 0], dtype=np.uint8)
cap.release()

cv2.circle(firstframe,  whiskercentre, 100, (200, 200, 200), 3, cv2.LINE_AA)
cv2.ellipse(firstframe, nosecentre,  (70,  50),   -60.0, 0, 360, (200,200,200), 3)
cv2.ellipse(firstframe, mouthcentre, (160, 60), mouthangle, 0, 360, (200,200,200), 3)
cv2.ellipse(firstframe, cheek_centre,(180, 100),   0.0,   0, 360, (200,200,200), 3)
for label, centre in [('whiskers', whiskercentre), ('nose', nosecentre),
                       ('mouth', mouthcentre), ('cheek', cheek_centre)]:
    cv2.putText(firstframe, label, (centre[0]-30, centre[1]),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (50,50,50), 2)

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(firstframe, cmap='gray')
ax.set_title('Face region masks')
ax.axis('off')
plt.tight_layout()
plt.show()


IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

### 5.2 Create binary masks and compute per-region motion energy

In [12]:
# ── Build boolean masks ───────────────────────────────────────────────────
canvas     = np.zeros([frame_height, frame_width])
whiskermask = cv2.circle(canvas.copy(),  whiskercentre, 100, (1,0,0), -1).astype(bool)
nosemask    = cv2.ellipse(canvas.copy(), nosecentre,  (70, 50),   -60.0, 0, 360, (1,0,0), -1).astype(bool)
mouthmask   = cv2.ellipse(canvas.copy(), mouthcentre, (160, 60), mouthangle, 0, 360, (1,0,0), -1).astype(bool)
cheekmask   = cv2.ellipse(canvas.copy(), cheek_centre,(180, 100),  0.0,   0, 360, (1,0,0), -1).astype(bool)
masks = np.array([nosemask, whiskermask, mouthmask, cheekmask]).astype('float32')
masks[masks == 0] = np.nan

# ── Compute motion energy per region ─────────────────────────────────────
mask_me = np.empty((sample_size, masks.shape[0]))
cap = cv2.VideoCapture(facevidpath)
ret, current_frame = cap.read()
previous_frame = current_frame.copy()
i = 0
with tqdm(total=sample_size, desc='Computing region motion energy') as pbar:
    while cap.isOpened():
        if not ret:
            break
        gray_curr = cv2.cvtColor(current_frame,  cv2.COLOR_BGR2GRAY)
        gray_prev = cv2.cvtColor(previous_frame, cv2.COLOR_BGR2GRAY)
        diff = cv2.absdiff(gray_curr, gray_prev)
        for j, mask in enumerate(masks):
            mask_me[i, j] = np.nanmean(diff * mask)
        previous_frame = current_frame.copy()
        ret, current_frame = cap.read()
        i += 1
        pbar.update(1)
        if i >= sample_size:
            break
cap.release()
print("✅ Motion energy computed.")


NameError: name 'whiskercentre' is not defined

### 5.3 Plot raw and filtered motion energy

In [ ]:
# ── Raw motion energy ─────────────────────────────────────────────────────
motion_energy_raw = pd.DataFrame(
    data=mask_me[:-1],  # drop last frame (incomplete diff)
    columns=['nose', 'whiskers', 'mouth', 'cheek']
)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

motion_energy_raw.astype(float).plot(ax=axes[0], alpha=0.8,
    ylabel='Raw motion energy [au]', title='Raw motion energy per face region')

# ── Filtered + Z-scored ───────────────────────────────────────────────────
motion_energy_filt = motion_energy_raw.apply(
    lambda x: pd.Series(smooth(x, window_len=75)).shift(periods=-37)
).dropna(axis=0)
motion_energy_z = motion_energy_filt.apply(zscore)

motion_energy_z.astype(float).plot(ax=axes[1], alpha=0.8,
    xlabel='Frame', ylabel='Motion energy [z-score]',
    title='Filtered & Z-scored motion energy per face region')

plt.tight_layout()
plt.show()


---
## 6. Optical Flow

Optical flow measures the direction and magnitude of pixel movement between frames using the Farneback dense optical flow algorithm. This captures richer movement information than motion energy:

- **Magnitude** → how much movement (similar to motion energy)
- **Angle** → direction of movement (oscillatory for whisking, unidirectional for head movement)

The angle signal is particularly useful for extracting whisking and sniffing frequencies in Section 7.


### 6.1 Visualise optical flow between two frames

In [14]:
# ── Extract two frames and compute optical flow between them ──────────────
time_points = [400, 402]  # frame indices to compare

cap = cv2.VideoCapture(facevidpath)
frames, frames_grey = [], []
for f in time_points:
    cap.set(1, f)
    _, frame = cap.read()
    frames.append(frame)
    frames_grey.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
cap.release()

frame_height, frame_width = frames[0].shape[:2]

flow = cv2.calcOpticalFlowFarneback(
    np.array(frames_grey[0]), np.array(frames_grey[1]),
    None, 0.5, 3, 15, 3, 5, 1.2, 0
)

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
axs[0].imshow(cv2.cvtColor(frames[0], cv2.COLOR_BGR2RGB))
axs[0].set_title(f'Frame {time_points[0]}')

step = 20
y_pts = np.arange(0, frame_height, step)
x_pts = np.arange(0, frame_width, step)
X, Y = np.meshgrid(x_pts, y_pts)
U = flow[::step, ::step, 0][:X.shape[0], :X.shape[1]]
V = flow[::step, ::step, 1][:X.shape[0], :X.shape[1]]
axs[1].quiver(X, Y, U, V)
axs[1].set_title('Optical flow vectors (quiver)')
axs[1].axis('equal')

rgb = flow_vis.flow_to_color(flow, convert_to_bgr=False)
axs[2].imshow(rgb)
axs[2].set_title('Optical flow (hue=direction, brightness=magnitude)')

for ax in axs:
    ax.axis('off')
plt.tight_layout()
plt.show()

### 6.2 Compute optical flow for a video segment

In [15]:
# ── Compute optical flow for whole frame (no face region masks needed) ────
frames_out = [400, 600]
flow_sample_size = frames_out[1] - frames_out[0]

flow_mag_all = np.empty(flow_sample_size)
flow_ang_all = np.empty(flow_sample_size)

cap = cv2.VideoCapture(facevidpath)
cap.set(1, frames_out[0])
ret, current_frame = cap.read()
previous_frame = current_frame.copy()
i = 0

with tqdm(total=flow_sample_size, desc='Computing optical flow') as pbar:
    while cap.isOpened() and i < flow_sample_size - 1:
        if not ret:
            break
        gray_curr = cv2.cvtColor(current_frame, cv2.COLOR_BGR2GRAY)
        gray_prev = cv2.cvtColor(previous_frame, cv2.COLOR_BGR2GRAY)
        flow = cv2.calcOpticalFlowFarneback(gray_prev, gray_curr, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        flow_mag_all[i] = np.mean(magnitude)
        flow_ang_all[i] = np.mean(angle)
        previous_frame = current_frame.copy()
        ret, current_frame = cap.read()
        i += 1
        pbar.update(1)
cap.release()

idx = range(frames_out[0], frames_out[0] + len(flow_ang_all) - 1)
ang = pd.Series(flow_ang_all[:-1], index=idx)
mag = pd.Series(flow_mag_all[:-1], index=idx)

fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax[0].plot(ang.index, ang)
ax[0].set(ylabel='Optical flow angle', title='Optical flow - whole frame')
ax[1].plot(mag.index, mag)
ax[1].set(xlabel='Frame', ylabel='Optical flow magnitude')
plt.tight_layout()
plt.show()
print("Done.")

Computing optical flow: 100%|█████████▉| 199/200 [00:07<00:00, 26.05it/s]


Done.


### 6.3 Optional: save optical flow video

Saves a side-by-side video of optical flow (color-coded) and the original frame, with face region outlines overlaid.


In [16]:
if save_videos:
    blend_gray_optflow = 0.2
    cap = cv2.VideoCapture(facevidpath)
    cap.set(1, frames_out[0])
    ret, current_frame = cap.read()
    previous_frame = current_frame.copy()
    fw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    fh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_vid = cv2.VideoWriter(
        os.path.join(out_dir, 'optical_flow_farneback.mp4'),
        fourcc, 30, (fw * 2, fh)
    )
    i = frames_out[0]
    with tqdm(total=flow_sample_size, desc='Saving optical flow video') as pbar:
        while cap.isOpened() and i < frames_out[1]:
            if not ret:
                break
            gray_curr = cv2.cvtColor(current_frame, cv2.COLOR_BGR2GRAY)
            gray_prev = cv2.cvtColor(previous_frame, cv2.COLOR_BGR2GRAY)
            flow = cv2.calcOpticalFlowFarneback(
                gray_prev, gray_curr, None, 0.5, 3, 15, 3, 5, 1.2, 0)
            rgb = flow_vis.flow_to_color(flow, convert_to_bgr=True)
            rgb = cv2.addWeighted(current_frame, blend_gray_optflow,
                                  rgb, 1 - blend_gray_optflow, 0)
            cv2.putText(rgb, str(i), (50, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            out_vid.write(np.concatenate((rgb, current_frame), axis=1))
            previous_frame = current_frame.copy()
            ret, current_frame = cap.read()
            i += 1
            pbar.update(1)
    cap.release()
    out_vid.release()
    print(f"✅ Optical flow video saved to {out_dir}/optical_flow_farneback.mp4")
    os.startfile(os.path.join(out_dir, 'optical_flow_farneback.mp4'))

Saving optical flow video: 100%|██████████| 200/200 [00:14<00:00, 13.80it/s]


✅ Optical flow video saved to C:\Users\ca1055\Downloads\mouseflow/optical_flow_farneback.mp4


---
## 7. Frequency Analysis (Whisking / Sniffing)

The optical flow angle signal from the whisker pad region oscillates at the whisking frequency (~5–15 Hz in mice). We extract this frequency by fitting a sinusoid to a rolling window of the signal.

**⚠️ Note on short recordings:** Sine fitting requires at least ~1–2 seconds of signal per window. For the 200-frame demo clip, results will be noisy. This works best on full session recordings (>10,000 frames).


In [22]:
# ── Sine fitting functions (credit: unsym, stackoverflow.com/a/42322656) ──

def fit_sin_w(yy):
    """Fit sinusoid and return angular frequency omega."""
    tt  = np.array(range(len(yy)))
    yy  = np.array(yy)
    ff  = np.fft.fftfreq(len(tt), (tt[1] - tt[0]))
    Fyy = abs(np.fft.fft(yy))
    guess = np.array([np.std(yy)*2**.5, 2.*np.pi*abs(ff[np.argmax(Fyy[1:])+1]), 0., np.mean(yy)])
    def sinfunc(t, A, w, p, c): return A * np.sin(w*t + p) + c
    try:
        popt, _ = optimize.curve_fit(sinfunc, tt, yy, p0=guess)
        return popt[1]
    except:
        return np.nan

def fit_sin_error(yy):
    """Fit sinusoid and return mean parameter uncertainty."""
    tt  = np.array(range(len(yy)))
    yy  = np.array(yy)
    ff  = np.fft.fftfreq(len(tt), (tt[1] - tt[0]))
    Fyy = abs(np.fft.fft(yy))
    guess = np.array([np.std(yy)*2**.5, 2.*np.pi*abs(ff[np.argmax(Fyy[1:])+1]), 0., np.mean(yy)])
    def sinfunc(t, A, w, p, c): return A * np.sin(w*t + p) + c
    try:
        _, pcov = optimize.curve_fit(sinfunc, tt, yy, p0=guess)
        return np.mean(np.sqrt(np.diag(pcov)))
    except:
        return np.nan

def fit_sin(yy):
    """Fit sinusoid and return all parameters including fitfunc."""
    tt  = np.array(range(len(yy)))
    yy  = np.array(yy)
    ff  = np.fft.fftfreq(len(tt), (tt[1] - tt[0]))
    Fyy = abs(np.fft.fft(yy))
    guess = np.array([np.std(yy)*2**.5, 2.*np.pi*abs(ff[np.argmax(Fyy[1:])+1]), 0., np.mean(yy)])
    def sinfunc(t, A, w, p, c): return A * np.sin(w*t + p) + c
    try:
        popt, pcov = optimize.curve_fit(sinfunc, tt, yy, p0=guess)
        A, w, p, c = popt
        return {'amp': A, 'omega': w, 'phase': p, 'offset': c,
                'freq': w/(2*np.pi), 'fitfunc': lambda t: A*np.sin(w*t+p)+c}
    except:
        return np.nan

print("✅ Sine fitting functions defined.")


✅ Sine fitting functions defined.


In [24]:
# ── Rolling sine fit on whole-frame optical flow angle ────────────────────
print("Fitting sine to optical flow angle...")
face_freq = freq_analysis(ang, fps=int(fps_vid))
face_freq.index = pd.Index(range(frames_out[0], frames_out[0] + len(face_freq)))

print(f"\nFrequency estimate summary:")
print(face_freq.describe().round(2))
print(f"NaN fraction: {face_freq.isnull().mean():.3f}")

face_freq.plot(figsize=(14, 4), ylabel='Frequency [Hz]', xlabel='Frame',
               title='Instantaneous oscillation frequency - whole frame')
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

Fitting sine to optical flow angle...

Frequency estimate summary:
count    197.00
mean       9.43
std        3.34
min        1.96
25%        6.61
50%        9.46
75%       12.22
max       16.29
dtype: float64
NaN fraction: 0.010


---
## Notes for your INCODE data

When running on your actual rig recordings (`vid3_*.mp4` or similar):

1. **Update paths** in Section 1.2 — change `facevidpath` and `out_dir`
2. **Check `facekey`** — the key string must appear in your video filename
3. **Fix `config.yaml`** if needed — run the cell below once per new video
4. **Frequency analysis** will work properly on full sessions (>10,000 frames)
5. **Face regions** (Section 5) require a full face view — skip if your camera only shows the eye


### Utility: Fix config.yaml for a new video

In [ ]:
# ── Run this if DLC gives a YAML ParserError ─────────────────────────────
# Updates the config.yaml video_sets entry to match your current video

import re

config_path = os.path.join(models_dir, 'MouseFace-Barnstedt-2019-08-21', 'config.yaml')

new_video_sets = f"""video_sets:
  {facevidpath.replace(chr(92), '/')}:
    crop: 0, {width}, 0, {height}
"""

with open(config_path, 'r') as f:
    content = f.read()

content_new = re.sub(r'video_sets:.*?(?=\nbodyparts:)', new_video_sets, content, flags=re.DOTALL)

with open(config_path, 'w') as f:
    f.write(content_new)

print(f"✅ config.yaml updated for: {facevidpath}")
print(f"   Crop: 0, {width}, 0, {height}")
